# Case Study 1: Univariate Time Series Forecasting — Air Passengers

## RNN vs LSTM vs GRU Comprehensive Comparison

---

### Objective
Predict monthly international airline passengers using three recurrent architectures:
- **Vanilla RNN** — simple recurrence, prone to vanishing gradients
- **LSTM** — gated memory cells (forget, input, output gates)
- **GRU** — simplified gating (reset, update gates)

### What You Will Learn
1. How to frame time series forecasting as a sequence-to-one problem
2. Sliding window approach for creating training samples
3. Building and comparing RNN, LSTM, GRU in PyTorch
4. Hyperparameter tuning with systematic search
5. Proper evaluation with naive baselines

### Dataset
- **Air Passengers**: 144 monthly observations (1949–1960)
- Classic Box-Jenkins dataset with trend + multiplicative seasonality

---
## 1. Environment Setup

In [ ]:
import random
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {DEVICE}')

# Plot style
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
COLORS = {'RNN': '#e74c3c', 'LSTM': '#2ecc71', 'GRU': '#3498db'}

---
## 2. Data Loading & Exploratory Data Analysis

In [ ]:
# Load dataset
df = pd.read_csv('Time_Series_Forecasting/air_passengers/air_passengers.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(10)

In [ ]:
# Convert decimal year to proper datetime
df['date'] = pd.date_range(start='1949-01', periods=len(df), freq='MS')
df = df.rename(columns={'value': 'passengers'})
df.set_index('date', inplace=True)
df = df[['passengers']]  # Keep only passengers column

print(f'Date range: {df.index[0]} to {df.index[-1]}')
print(f'\nSummary Statistics:')
df.describe()

In [ ]:
# Time series plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['passengers'], color='#2c3e50', linewidth=1.5)
ax.set_title('Monthly International Airline Passengers (1949-1960)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Passengers (thousands)')
ax.axvline(df.index[int(len(df)*0.8)], color='red', linestyle='--', alpha=0.7, label='Train/Test Split')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Seasonal decomposition
decomposition = seasonal_decompose(df['passengers'], model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomposition.observed.plot(ax=axes[0], title='Observed', color='#2c3e50')
decomposition.trend.plot(ax=axes[1], title='Trend', color='#e74c3c')
decomposition.seasonal.plot(ax=axes[2], title='Seasonal', color='#2ecc71')
decomposition.resid.plot(ax=axes[3], title='Residual', color='#3498db')
plt.suptitle('Multiplicative Decomposition', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution and Autocorrelation
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Histogram
axes[0].hist(df['passengers'], bins=20, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of Passengers')
axes[0].set_xlabel('Passengers (thousands)')

# ACF
plot_acf(df['passengers'], lags=40, ax=axes[1], color='#2ecc71')
axes[1].set_title('Autocorrelation Function (ACF)')

# PACF
plot_pacf(df['passengers'], lags=40, ax=axes[2], color='#e74c3c', method='ywm')
axes[2].set_title('Partial Autocorrelation Function (PACF)')

plt.tight_layout()
plt.show()

print('Key observations:')
print('- ACF shows slow decay -> non-stationary (trend)')
print('- ACF peaks at lags 12, 24, 36 -> strong yearly seasonality')
print('- Multiplicative seasonality: amplitude grows with level')

---
## 3. Data Preprocessing

In [ ]:
# Extract raw values
data = df['passengers'].values.astype(np.float32).reshape(-1, 1)

# Train/Test split (80/20 — chronological, NO shuffle)
train_size = int(len(data) * 0.8)
train_data = data[:train_size]
test_data = data[train_size:]

print(f'Train samples: {len(train_data)} (months 1-{train_size})')
print(f'Test samples:  {len(test_data)} (months {train_size+1}-{len(data)})')

# Scale using MinMaxScaler (fit on train only!)
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

# Combine for sliding window (we need train context to predict first test point)
all_scaled = np.concatenate([train_scaled, test_scaled], axis=0)

print(f'\nScaled range: [{all_scaled.min():.4f}, {all_scaled.max():.4f}]')

In [ ]:
def create_sequences(data, seq_length):
    """Create sliding window sequences for time series forecasting.
    
    Args:
        data: numpy array of shape (n_samples, n_features)
        seq_length: number of past timesteps to use as input
    
    Returns:
        X: tensor of shape (n_windows, seq_length, n_features)
        y: tensor of shape (n_windows, n_features)
    """
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return torch.FloatTensor(np.array(X)), torch.FloatTensor(np.array(y))


class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Default sequence length
SEQ_LENGTH = 12  # 12 months lookback (one full year)

# Create train sequences
X_train, y_train = create_sequences(train_scaled, SEQ_LENGTH)
# Create test sequences (using tail of train + test for context)
test_input = all_scaled[train_size - SEQ_LENGTH:]
X_test, y_test = create_sequences(test_input, SEQ_LENGTH)

print(f'Training sequences: X={X_train.shape}, y={y_train.shape}')
print(f'Test sequences:     X={X_test.shape}, y={y_test.shape}')

In [ ]:
# Visualize a sample sliding window
fig, ax = plt.subplots(figsize=(10, 4))
sample_idx = 50
window = X_train[sample_idx, :, 0].numpy()
target = y_train[sample_idx, 0].numpy()

ax.plot(range(SEQ_LENGTH), window, 'o-', color='#3498db', label='Input window', markersize=6)
ax.plot(SEQ_LENGTH, target, 's', color='#e74c3c', markersize=10, label='Target', zorder=5)
ax.axvline(SEQ_LENGTH - 0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Time step')
ax.set_ylabel('Scaled value')
ax.set_title(f'Sliding Window Example (window size = {SEQ_LENGTH})')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Model Architecture

### Architecture Diagram
```
Input: (batch, seq_len=12, features=1)
       |
  +--------+   +--------+   +--------+         +--------+
  | RNN /  |-->| RNN /  |-->| RNN /  |--> ... ->| RNN /  |--> h_T
  | LSTM / |   | LSTM / |   | LSTM / |         | LSTM / |
  | GRU    |   | GRU    |   | GRU    |         | GRU    |
  +--------+   +--------+   +--------+         +--------+
    t=1          t=2          t=3                 t=12
                                                   |
                                              [Linear Layer]
                                                   |
                                              Output: (batch, 1)
                                        (predicted next value)
```

In [ ]:
class SequenceForecaster(nn.Module):
    """Unified RNN/LSTM/GRU model for sequence forecasting."""
    
    SUPPORTED_TYPES = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}
    
    def __init__(self, model_type, input_size, hidden_size, output_size,
                 num_layers=1, dropout=0.0, bidirectional=False):
        super().__init__()
        assert model_type in self.SUPPORTED_TYPES, f'model_type must be one of {list(self.SUPPORTED_TYPES.keys())}'
        
        self.model_type = model_type
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        
        rnn_cls = self.SUPPORTED_TYPES[model_type]
        self.rnn = rnn_cls(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        
        fc_input_size = hidden_size * (2 if bidirectional else 1)
        self.fc = nn.Linear(fc_input_size, output_size)
    
    def forward(self, x):
        # x: (batch, seq_len, input_size)
        rnn_out, _ = self.rnn(x)      # rnn_out: (batch, seq_len, hidden_size * num_directions)
        last_hidden = rnn_out[:, -1, :]  # Take last timestep
        output = self.fc(last_hidden)    # (batch, output_size)
        return output
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
# Compare parameter counts
param_comparison = []
for model_type in ['RNN', 'LSTM', 'GRU']:
    model = SequenceForecaster(model_type, input_size=1, hidden_size=64, output_size=1, num_layers=2)
    params = model.count_parameters()
    param_comparison.append({'Model': model_type, 'Parameters': params})
    print(f'{model_type:5s}: {params:,} parameters')

param_df = pd.DataFrame(param_comparison)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(param_df['Model'], param_df['Parameters'], 
              color=[COLORS[m] for m in param_df['Model']], edgecolor='white', width=0.5)
for bar, val in zip(bars, param_df['Parameters']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{val:,}', ha='center', fontweight='bold')
ax.set_title('Parameter Count Comparison (hidden=64, layers=2)', fontsize=13)
ax.set_ylabel('Trainable Parameters')
plt.tight_layout()
plt.show()

print(f'\nLSTM has ~4x parameters of RNN (3 extra gates)')
print(f'GRU has ~3x parameters of RNN (2 gates vs 0)')
print(f'GRU has ~75% parameters of LSTM (2 gates vs 3)')

---
## 5. Training Infrastructure

In [ ]:
def train_model(model, train_loader, X_val, y_val, epochs, lr,
                device=DEVICE, patience=10, clip_grad=1.0, verbose=True):
    """Train a model with early stopping and gradient clipping.
    
    Returns:
        history: dict with 'train_loss' and 'val_loss' lists
        training_time: total training time in seconds
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=False)
    
    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    
    X_val_dev = X_val.to(device)
    y_val_dev = y_val.to(device)
    
    start_time = time.time()
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_losses = []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
            train_losses.append(loss.item())
        
        # Validation
        model.eval()
        with torch.no_grad():
            val_output = model(X_val_dev)
            val_loss = criterion(val_output, y_val_dev).item()
        
        avg_train = np.mean(train_losses)
        history['train_loss'].append(avg_train)
        history['val_loss'].append(val_loss)
        scheduler.step(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print(f'  Early stopping at epoch {epoch+1}')
                break
        
        if verbose and (epoch + 1) % 20 == 0:
            print(f'  Epoch {epoch+1:3d}/{epochs} | Train Loss: {avg_train:.6f} | Val Loss: {val_loss:.6f}')
    
    training_time = time.time() - start_time
    
    # Restore best model
    if best_state is not None:
        model.load_state_dict(best_state)
        model = model.to(device)
    
    return history, training_time

In [ ]:
def evaluate_forecast(model, X_test, y_test, scaler, device=DEVICE):
    """Evaluate a forecasting model and return metrics + predictions."""
    model.eval()
    with torch.no_grad():
        predictions = model(X_test.to(device)).cpu().numpy()
    
    # Inverse scale
    pred_original = scaler.inverse_transform(predictions)
    actual_original = scaler.inverse_transform(y_test.numpy())
    
    # Metrics
    mse = mean_squared_error(actual_original, pred_original)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actual_original, pred_original)
    r2 = r2_score(actual_original, pred_original)
    mape = np.mean(np.abs((actual_original - pred_original) / actual_original)) * 100
    
    metrics = {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': f'{mape:.2f}%', 'R2': r2}
    return metrics, pred_original.flatten(), actual_original.flatten()

---
## 6. Hyperparameter Tuning

We perform a systematic search over key hyperparameters for each model type.

In [ ]:
# Hyperparameter search space
SEARCH_CONFIGS = [
    {'hidden_size': 32,  'num_layers': 1, 'lr': 0.01,  'dropout': 0.0, 'seq_length': 12},
    {'hidden_size': 64,  'num_layers': 1, 'lr': 0.005, 'dropout': 0.0, 'seq_length': 12},
    {'hidden_size': 64,  'num_layers': 2, 'lr': 0.005, 'dropout': 0.1, 'seq_length': 12},
    {'hidden_size': 128, 'num_layers': 1, 'lr': 0.001, 'dropout': 0.0, 'seq_length': 12},
    {'hidden_size': 128, 'num_layers': 2, 'lr': 0.001, 'dropout': 0.2, 'seq_length': 12},
    {'hidden_size': 64,  'num_layers': 1, 'lr': 0.005, 'dropout': 0.0, 'seq_length': 24},
    {'hidden_size': 128, 'num_layers': 2, 'lr': 0.001, 'dropout': 0.1, 'seq_length': 24},
    {'hidden_size': 256, 'num_layers': 1, 'lr': 0.001, 'dropout': 0.0, 'seq_length': 12},
]

BATCH_SIZE = 16
TUNING_EPOCHS = 100

print(f'Search space: {len(SEARCH_CONFIGS)} configs x 3 model types = {len(SEARCH_CONFIGS)*3} total trials')

In [ ]:
tuning_results = []

for model_type in ['RNN', 'LSTM', 'GRU']:
    print(f'\n{"="*60}')
    print(f'Tuning {model_type}')
    print(f'{"="*60}')
    
    for i, config in enumerate(SEARCH_CONFIGS):
        # Create sequences with this config's seq_length
        X_tr, y_tr = create_sequences(train_scaled, config['seq_length'])
        test_input = all_scaled[train_size - config['seq_length']:]
        X_te, y_te = create_sequences(test_input, config['seq_length'])
        
        train_dataset = TimeSeriesDataset(X_tr, y_tr)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        
        # Build model
        model = SequenceForecaster(
            model_type=model_type,
            input_size=1,
            hidden_size=config['hidden_size'],
            output_size=1,
            num_layers=config['num_layers'],
            dropout=config['dropout']
        )
        
        # Train
        history, train_time = train_model(
            model, train_loader, X_te, y_te,
            epochs=TUNING_EPOCHS, lr=config['lr'], verbose=False
        )
        
        best_val = min(history['val_loss'])
        tuning_results.append({
            'model_type': model_type,
            'config_id': i,
            **config,
            'best_val_loss': best_val,
            'train_time': train_time,
            'epochs_run': len(history['val_loss']),
            'params': model.count_parameters()
        })
        
        print(f'  Config {i+1}/{len(SEARCH_CONFIGS)}: h={config["hidden_size"]}, '
              f'L={config["num_layers"]}, lr={config["lr"]}, '
              f'seq={config["seq_length"]} -> val_loss={best_val:.6f} ({train_time:.1f}s)')

tuning_df = pd.DataFrame(tuning_results)
print(f'\nTotal trials: {len(tuning_df)}')

In [ ]:
# Find best config per model type
best_configs = {}
print('Best configurations per model type:')
print('=' * 80)

for model_type in ['RNN', 'LSTM', 'GRU']:
    subset = tuning_df[tuning_df['model_type'] == model_type]
    best_row = subset.loc[subset['best_val_loss'].idxmin()]
    best_configs[model_type] = best_row.to_dict()
    print(f"\n{model_type}:")
    print(f"  hidden_size={int(best_row['hidden_size'])}, num_layers={int(best_row['num_layers'])}, "
          f"lr={best_row['lr']}, dropout={best_row['dropout']}, seq_length={int(best_row['seq_length'])}")
    print(f"  Best val loss: {best_row['best_val_loss']:.6f}")

In [ ]:
# Tuning results heatmap
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, model_type in zip(axes, ['RNN', 'LSTM', 'GRU']):
    subset = tuning_df[tuning_df['model_type'] == model_type]
    pivot_data = subset.pivot_table(
        values='best_val_loss', 
        index='hidden_size', 
        columns='num_layers',
        aggfunc='min'
    )
    sns.heatmap(pivot_data, annot=True, fmt='.5f', cmap='YlOrRd_r', ax=ax)
    ax.set_title(f'{model_type} — Val Loss by Hidden Size & Layers')

plt.suptitle('Hyperparameter Tuning Results', fontsize=14)
plt.tight_layout()
plt.show()

---
## 7. Final Model Training & Comparison

In [ ]:
# Retrain best models with more epochs
FINAL_EPOCHS = 200
final_models = {}
final_histories = {}
final_times = {}

for model_type in ['RNN', 'LSTM', 'GRU']:
    print(f'\nTraining final {model_type} model...')
    cfg = best_configs[model_type]
    seq_len = int(cfg['seq_length'])
    
    # Create sequences with best seq_length
    X_tr, y_tr = create_sequences(train_scaled, seq_len)
    test_input = all_scaled[train_size - seq_len:]
    X_te, y_te = create_sequences(test_input, seq_len)
    
    train_dataset = TimeSeriesDataset(X_tr, y_tr)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    model = SequenceForecaster(
        model_type=model_type,
        input_size=1,
        hidden_size=int(cfg['hidden_size']),
        output_size=1,
        num_layers=int(cfg['num_layers']),
        dropout=cfg['dropout']
    )
    
    history, train_time = train_model(
        model, train_loader, X_te, y_te,
        epochs=FINAL_EPOCHS, lr=cfg['lr'], verbose=True
    )
    
    final_models[model_type] = model
    final_histories[model_type] = history
    final_times[model_type] = train_time
    
    print(f'  {model_type} done: {len(history["val_loss"])} epochs, {train_time:.1f}s')

In [ ]:
# Training curves comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

for model_type in ['RNN', 'LSTM', 'GRU']:
    h = final_histories[model_type]
    ax1.plot(h['train_loss'], label=model_type, color=COLORS[model_type], linewidth=1.5)
    ax2.plot(h['val_loss'], label=model_type, color=COLORS[model_type], linewidth=1.5)

ax1.set_title('Training Loss', fontsize=13)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.legend(fontsize=11)
ax1.set_yscale('log')

ax2.set_title('Validation Loss', fontsize=13)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MSE Loss')
ax2.legend(fontsize=11)
ax2.set_yscale('log')

plt.suptitle('Training Curves: RNN vs LSTM vs GRU', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate all models and collect predictions
all_metrics = {}
all_predictions = {}

for model_type in ['RNN', 'LSTM', 'GRU']:
    cfg = best_configs[model_type]
    seq_len = int(cfg['seq_length'])
    test_input = all_scaled[train_size - seq_len:]
    X_te, y_te = create_sequences(test_input, seq_len)
    
    metrics, preds, actuals = evaluate_forecast(
        final_models[model_type], X_te, y_te, scaler
    )
    metrics['Training Time'] = f'{final_times[model_type]:.1f}s'
    metrics['Parameters'] = final_models[model_type].count_parameters()
    
    all_metrics[model_type] = metrics
    all_predictions[model_type] = preds

# Naive baseline: predict last known value
naive_preds = data[train_size-1:train_size+len(actuals)-1].flatten()
naive_mse = mean_squared_error(actuals, naive_preds)
naive_rmse = np.sqrt(naive_mse)
naive_mae = mean_absolute_error(actuals, naive_preds)
naive_mape = np.mean(np.abs((actuals - naive_preds) / actuals)) * 100
naive_r2 = r2_score(actuals, naive_preds)
all_metrics['Naive Baseline'] = {
    'MSE': naive_mse, 'RMSE': naive_rmse, 'MAE': naive_mae,
    'MAPE': f'{naive_mape:.2f}%', 'R2': naive_r2,
    'Training Time': '0s', 'Parameters': 0
}

# Display comparison table
metrics_df = pd.DataFrame(all_metrics).T
for col in ['MSE', 'RMSE', 'MAE', 'R2']:
    metrics_df[col] = metrics_df[col].apply(lambda x: f'{x:.4f}' if isinstance(x, float) else x)

print('\n' + '='*80)
print('FINAL COMPARISON TABLE')
print('='*80)
metrics_df

In [ ]:
# Predictions overlay
test_dates = df.index[train_size:]

fig, ax = plt.subplots(figsize=(16, 6))

# Plot full series faded
ax.plot(df.index[:train_size], df['passengers'].values[:train_size], 
        color='#95a5a6', alpha=0.5, label='Training Data')

# Plot actual test
ax.plot(test_dates[:len(actuals)], actuals, 
        color='black', linewidth=2, label='Actual', marker='o', markersize=4)

# Plot each model's predictions
for model_type in ['RNN', 'LSTM', 'GRU']:
    ax.plot(test_dates[:len(all_predictions[model_type])], all_predictions[model_type],
            color=COLORS[model_type], linewidth=1.5, label=model_type, 
            linestyle='--', marker='s', markersize=3, alpha=0.8)

ax.axvline(df.index[train_size], color='gray', linestyle=':', alpha=0.7)
ax.set_title('Test Set Predictions: RNN vs LSTM vs GRU', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Passengers (thousands)')
ax.legend(loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Residual analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, model_type in zip(axes, ['RNN', 'LSTM', 'GRU']):
    residuals = actuals - all_predictions[model_type]
    ax.bar(range(len(residuals)), residuals, color=COLORS[model_type], alpha=0.7, edgecolor='white')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'{model_type} Residuals')
    ax.set_xlabel('Test Sample Index')
    ax.set_ylabel('Residual (Actual - Predicted)')
    ax.text(0.02, 0.98, f'Mean: {np.mean(residuals):.1f}\nStd: {np.std(residuals):.1f}',
            transform=ax.transAxes, verticalalignment='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Residual Analysis', fontsize=14)
plt.tight_layout()
plt.show()

---
## 8. Analysis & Insights

In [ ]:
# Sequence length ablation study
print('Sequence Length Ablation Study')
print('='*60)

seq_lengths = [3, 6, 12, 18, 24, 36]
ablation_results = []

for seq_len in seq_lengths:
    X_tr, y_tr = create_sequences(train_scaled, seq_len)
    test_input = all_scaled[train_size - seq_len:]
    X_te, y_te = create_sequences(test_input, seq_len)
    
    train_dataset = TimeSeriesDataset(X_tr, y_tr)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    for model_type in ['RNN', 'LSTM', 'GRU']:
        model = SequenceForecaster(
            model_type=model_type, input_size=1, hidden_size=64,
            output_size=1, num_layers=1
        )
        history, _ = train_model(model, train_loader, X_te, y_te,
                                  epochs=100, lr=0.005, verbose=False)
        best_val = min(history['val_loss'])
        ablation_results.append({
            'seq_length': seq_len, 'model_type': model_type, 'val_loss': best_val
        })
    print(f'  seq_length={seq_len:2d} done')

ablation_df = pd.DataFrame(ablation_results)

In [ ]:
# Plot ablation results
fig, ax = plt.subplots(figsize=(10, 5))

for model_type in ['RNN', 'LSTM', 'GRU']:
    subset = ablation_df[ablation_df['model_type'] == model_type]
    ax.plot(subset['seq_length'], subset['val_loss'], 'o-',
            color=COLORS[model_type], label=model_type, linewidth=2, markersize=8)

ax.set_xlabel('Sequence Length (months)', fontsize=12)
ax.set_ylabel('Best Validation Loss (MSE)', fontsize=12)
ax.set_title('Effect of Sequence Length on Model Performance', fontsize=14)
ax.legend(fontsize=11)
ax.set_xticks(seq_lengths)
plt.tight_layout()
plt.show()

print('Observations:')
print('- Too short (3 months): insufficient context for seasonal patterns')
print('- 12 months (1 year): captures full seasonal cycle')
print('- Longer sequences: diminishing returns, possibly worse for RNN (vanishing gradients)')

In [ ]:
# Inference speed comparison
inference_times = {}

for model_type in ['RNN', 'LSTM', 'GRU']:
    cfg = best_configs[model_type]
    seq_len = int(cfg['seq_length'])
    test_input = all_scaled[train_size - seq_len:]
    X_te, y_te = create_sequences(test_input, seq_len)
    
    model = final_models[model_type].to(DEVICE)
    model.eval()
    
    # Warm up
    with torch.no_grad():
        _ = model(X_te.to(DEVICE))
    
    # Time 100 inference runs
    start = time.time()
    with torch.no_grad():
        for _ in range(100):
            _ = model(X_te.to(DEVICE))
    elapsed = (time.time() - start) / 100
    inference_times[model_type] = elapsed * 1000  # ms

print('\nInference Speed (ms per batch):')
for mt, t in inference_times.items():
    print(f'  {mt}: {t:.2f} ms')

---
## 9. Key Takeaways

### RNN vs LSTM vs GRU — Summary

| Aspect | RNN | LSTM | GRU |
|--------|-----|------|-----|
| **Parameters** | Fewest | Most (~4x RNN) | Middle (~3x RNN) |
| **Vanishing Gradient** | Severe | Mitigated by gates | Mitigated by gates |
| **Long Dependencies** | Poor | Best | Good |
| **Training Speed** | Fastest per epoch | Slowest per epoch | Middle |
| **Best For** | Very short sequences | Long sequences, complex patterns | Good default choice |

### For This Dataset (Air Passengers)
- With only 144 data points and 12-month seasonality, even simple models can capture the pattern
- LSTM and GRU offer marginal gains over RNN because the sequence dependency is short (12 months)
- The **naive baseline** is surprisingly strong — always compare against it!
- **GRU** often provides the best accuracy-to-complexity tradeoff on small datasets

### Extensions
- Try this on **Sunspot Activity** data (309 years, 11-year cycles — longer dependencies favor LSTM)
- Add **bidirectional** processing (helps when full sequence is available)
- Experiment with **stacked layers** (2-3 layers) for more capacity
- Compare with **SARIMA** or **Prophet** from `statsmodels`/`prophet`

In [ ]:
# Final summary visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. RMSE comparison
rmse_vals = [float(all_metrics[m]['RMSE']) for m in ['RNN', 'LSTM', 'GRU', 'Naive Baseline']]
colors_bar = [COLORS['RNN'], COLORS['LSTM'], COLORS['GRU'], '#95a5a6']
bars = axes[0].bar(['RNN', 'LSTM', 'GRU', 'Naive'], rmse_vals, color=colors_bar, edgecolor='white')
for bar, val in zip(bars, rmse_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.2f}', ha='center', fontweight='bold')
axes[0].set_title('RMSE Comparison', fontsize=13)
axes[0].set_ylabel('RMSE')

# 2. Training time
time_vals = [final_times[m] for m in ['RNN', 'LSTM', 'GRU']]
axes[1].bar(['RNN', 'LSTM', 'GRU'], time_vals,
            color=[COLORS[m] for m in ['RNN', 'LSTM', 'GRU']], edgecolor='white')
axes[1].set_title('Training Time', fontsize=13)
axes[1].set_ylabel('Seconds')

# 3. Parameters
param_vals = [all_metrics[m]['Parameters'] for m in ['RNN', 'LSTM', 'GRU']]
axes[2].bar(['RNN', 'LSTM', 'GRU'], param_vals,
            color=[COLORS[m] for m in ['RNN', 'LSTM', 'GRU']], edgecolor='white')
axes[2].set_title('Parameter Count', fontsize=13)
axes[2].set_ylabel('Parameters')

plt.suptitle('Final Summary: RNN vs LSTM vs GRU on Air Passengers', fontsize=14)
plt.tight_layout()
plt.show()

print('\nNotebook complete! Proceed to Notebook 02 for multivariate forecasting.')